In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

### Handle class imbalance

In [5]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

### Track Experiments

In [6]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [8]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [9]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [10]:
# Initialize MLflow
mlflow.set_experiment("Anomaly Detection")
mlflow.set_tracking_uri("http://localhost:5000")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")

2026/01/29 13:48:10 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/01/29 13:48:10 INFO mlflow.store.db.utils: Updating database tables
2026/01/29 13:48:10 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/29 13:48:10 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/29 13:48:10 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/01/29 13:48:10 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/01/29 13:48:10 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection' does not exist. Creating a new experiment.
2026/01/29 13:48:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://localhost:5000/#/experiments/2/runs/55190de53f4140d9a93834a7b3b3771f
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/01/29 13:48:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/2/runs/0651711d2ae245f79e024f40d517a31d
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/01/29 13:49:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://localhost:5000/#/experiments/2/runs/0b343b00eae741f0bb661a712b179fb5
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/01/29 13:49:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: http://localhost:5000/#/experiments/2/runs/39de9e284e13430abe1b6db277d10b87
🧪 View experiment at: http://localhost:5000/#/experiments/2


### Register the Model

In [14]:

model_name = "XGB-SmoteClassifier"
run_id = input("Enter Run Id: ").strip()

# This must match the artifact_path used in log_model()
model_uri = f"runs:/{run_id}/model"

mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)


Enter Run Id:  39de9e284e13430abe1b6db277d10b87


Registered model 'XGB-SmoteClassifier' already exists. Creating a new version of this model...
2026/01/29 14:04:03 WARNING mlflow.tracking._model_registry.fluent: Run with id 39de9e284e13430abe1b6db277d10b87 has no artifacts at artifact path 'model', registering model based on models:/m-f1a818744c8a433cbf6970ed3cfb024f instead
2026/01/29 14:04:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-SmoteClassifier, version 1
Created version '1' of model 'XGB-SmoteClassifier'.


<ModelVersion: aliases=[], creation_timestamp=1769675643547, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1769675643547, metrics=None, model_id=None, name='XGB-SmoteClassifier', params=None, run_id='39de9e284e13430abe1b6db277d10b87', run_link='', source='models:/m-f1a818744c8a433cbf6970ed3cfb024f', status='READY', status_message=None, tags={}, user_id='', version='1'>

### Load the Model

In [15]:
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

### Transition the Model to Production

In [17]:
current_model_uri = f"models:/{model_name}@champion"
production_model_name = "anomaly-detection-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

Successfully registered model 'anomaly-detection-prod'.
Copied version '1' of model 'XGB-SmoteClassifier' to version '1' of model 'anomaly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1769675935634, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1769675935634, metrics=None, model_id=None, name='anomaly-detection-prod', params=None, run_id='39de9e284e13430abe1b6db277d10b87', run_link='', source='models:/XGB-SmoteClassifier/1', status='READY', status_message=None, tags={}, user_id='', version='1'>

In [23]:
model_version = 1
prod_model_uri = f"models:/{production_model_name}@great"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])